# PDF Load Test 5000 - DO (Data Owner) v4

Run this notebook in a Colab session logged in as **test1@openmined.org**.

Run cells in order, coordinating with the DS notebook.

PDFs are stored in `My Drive/syft_test_data/pdfs_5000/` — outside SyftBox so cleanup won't touch them.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BRANCH = "feature/support-aisi-pdf-download"
REPO = "https://github.com/OpenMined/syft-client.git"

!pip install -q \
  "syft-bg @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-bg" \
  "syft-job @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-job" \
  "syft-dataset @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-datasets" \
  "syft-permissions @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-permissions" \
  "syft-perm @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-perm" \
  "syft-client @ git+{REPO}@{BRANCH}"

import syft_client as sc
print(f"syft_client version: {sc.__version__}")

In [ ]:
from pathlib import Path
import shutil
import time
import json

DO_EMAIL = "test1@openmined.org"
DS_EMAIL = "test2@openmined.org"

# Source PDFs (existing 200) and target folder — both OUTSIDE SyftBox
SOURCE_PDF_FOLDER = Path("/content/drive/MyDrive/SyftBox/syft_test_pdfs_1000")
PDF_FOLDER = Path("/content/drive/MyDrive/syft_test_data/pdfs_5000")
DATASET_NAME = "PDF_Test_5000"
JOB_NAME = "pdf_analysis_5000"
TARGET_COUNT = 5000

---
## DO Login

In [ ]:
do_client = sc.login_do(email=DO_EMAIL)
print(f"DO: {do_client.email}")
print(f"SyftBox: {do_client.syftbox_folder}")

---
## Generate 5000 PDFs

Copies the existing 200 PDFs with renames to make 5000. Skip if already done.
Stored in `My Drive/syft_test_data/pdfs_5000/` — safe from SyftBox cleanup.

In [ ]:
PDF_FOLDER.mkdir(parents=True, exist_ok=True)

existing = sorted(PDF_FOLDER.glob("*.pdf"))
print(f"Target folder: {PDF_FOLDER}")
print(f"Existing PDFs: {len(existing)}")

if len(existing) >= TARGET_COUNT:
    print(f"Already have {len(existing)} PDFs, skipping")
else:
    # Get source PDFs
    source_pdfs = sorted(SOURCE_PDF_FOLDER.glob("*.pdf"))
    assert len(source_pdfs) > 0, f"No source PDFs at {SOURCE_PDF_FOLDER}"
    print(f"Source PDFs: {len(source_pdfs)}")

    # Copy with renames: batch_00_original.pdf, batch_01_original.pdf, ...
    num_batches = (TARGET_COUNT // len(source_pdfs)) + 1
    count = len(existing)
    for batch in range(num_batches):
        if count >= TARGET_COUNT:
            break
        for src in source_pdfs:
            if count >= TARGET_COUNT:
                break
            dst = PDF_FOLDER / f"batch{batch:02d}_{src.name}"
            if not dst.exists():
                shutil.copy2(src, dst)
                count += 1
        if batch % 5 == 0:
            print(f"  Batch {batch}: {count}/{TARGET_COUNT} PDFs")

    final_count = len(list(PDF_FOLDER.glob("*.pdf")))
    total_size = sum(f.stat().st_size for f in PDF_FOLDER.glob("*.pdf"))
    print(f"Done: {final_count} PDFs, {total_size / (1024*1024):.1f} MB total")

---
## Create Dataset

In [ ]:
pdf_files = sorted(PDF_FOLDER.glob("*.pdf"))
print(f"PDFs found: {len(pdf_files)}")
assert len(pdf_files) >= TARGET_COUNT, f"Expected {TARGET_COUNT}, found {len(pdf_files)}"

# Cleanup existing dataset (allows re-runs)
for p in [
    do_client.syftbox_folder / do_client.email / "public" / "syft_datasets" / DATASET_NAME,
    do_client.syftbox_folder / "private" / "syft_datasets" / DATASET_NAME,
]:
    if p.exists():
        shutil.rmtree(p)
        print(f"Cleaned up: {p}")

In [ ]:
# Create manifest and readme
manifest_path = Path("/content/manifest.csv")
csv_lines = ["filename,doc_id,category"]
for i, pdf in enumerate(pdf_files):
    csv_lines.append(f"{pdf.name},DOC{i:05d},{['report','invoice','contract','memo'][i%4]}")
manifest_path.write_text("\n".join(csv_lines))

readme_path = Path("/content/readme.md")
readme_path.write_text(f"# PDF Test\n\n{len(pdf_files)} PDFs for load testing\n")
print(f"Manifest: {len(pdf_files)} entries")

In [ ]:
start = time.time()
do_client.create_dataset(
    name=DATASET_NAME,
    mock_path=str(manifest_path),
    private_path=str(PDF_FOLDER),
    summary=f"Load test: {len(pdf_files)} PDFs",
    readme_path=str(readme_path),
    tags=["test", "pdf"],
    users=[DS_EMAIL],
    upload_private=False,
    sync=True
)
print(f"Dataset created in {time.time() - start:.1f}s")

# Verify
private_ds = do_client.syftbox_folder / "private" / "syft_datasets" / DATASET_NAME
print(f"Private dataset exists: {private_ds.exists()}")
if private_ds.exists():
    print(f"Private PDFs: {len(list(private_ds.rglob('*.pdf')))}")

---
## Approve Peer

**Wait for DS to add peer first**, then run this cell.

In [ ]:
do_client.sync()
do_client.load_peers()

print(f"Approved peers: {[p.email for p in do_client.peers if p.state.value == 'accepted']}")

if hasattr(do_client, 'peer_manager') and do_client.peer_manager:
    pending = getattr(do_client.peer_manager, 'pending_peers', [])
    print(f"Pending requests: {[p.email for p in pending]}")
    for req in pending:
        do_client.approve_peer_request(req.email)
        print(f"Approved: {req.email}")

do_client.sync()
print("Done")

---
## Approve + Run Job

**Wait for DS to submit job first**, then run these cells.

In [ ]:
do_client.sync()

# Debug: verify inbox folder and messages from DS
conn = do_client.peer_manager.connection_router.connection_for_send_message()
conn.do_inbox_folder_id_cache = {}
inbox_id = conn._get_inbox_folder_id_as_do(DS_EMAIL)
print(f"DO inbox folder ID (from DS): {inbox_id}")
if inbox_id:
    files_in_inbox = conn.get_file_metadatas_from_folder(inbox_id)
    print(f"Files in inbox: {len(files_in_inbox)}")
    for f in files_in_inbox:
        print(f"  {f['name']} (id={f['id']})")
else:
    print("WARNING: Inbox folder not found! DS may not have shared it properly.")

print("\nDO's jobs:")
for job in do_client.jobs:
    print(f"  {job.name} | status={job.status} | submitted_by={getattr(job, 'submitted_by', '?')}")

for job in do_client.jobs:
    if job.name == JOB_NAME and job.status == "inbox":
        job.approve()
        print(f"Approved: {job.name}")

In [ ]:
start = time.time()
do_client.process_approved_jobs(stream_output=True, timeout=300)
print(f"\nExecution took {time.time() - start:.1f}s")

# Verify on DO side
do_job_dir = do_client.syftbox_folder / do_client.email / "app_data" / "job" / JOB_NAME
print(f"'done' marker: {(do_job_dir / 'done').exists()}")

if (do_job_dir / 'stdout.txt').exists():
    print(f"stdout: {(do_job_dir / 'stdout.txt').read_text()}")

outputs_dir = do_job_dir / "outputs"
if outputs_dir.exists():
    analysis = outputs_dir / "analysis.json"
    if analysis.exists():
        print(f"analysis.json: {json.loads(analysis.read_text())}")

In [ ]:
# Push results to DS
do_client.sync()
print("Results pushed")

for job in do_client.jobs:
    if job.name == JOB_NAME:
        print(f"DO job status: {job.status}")